### Averaging out replicates

In [1]:
import os
import re
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bars

def extract_base(col_name):
    """
    Extracts the base condition name from a column name by removing a replicate marker.
    Recognizes patterns like:
      - rep1, rep 1, rep_1, replicate 1, replicate_1, Replicate, etc.
    even when followed by extra text.
    """
    pattern = re.compile(r"^(.*?)\s*(rep(?:licate)?\s*[_-]?\s*\d+)(.*)$", re.IGNORECASE)
    m = pattern.match(col_name)
    if m:
        # Concatenate the part before the replicate marker with the part after it.
        base = (m.group(1) + m.group(3)).strip()
        if base:
            return base
    return col_name

def format_float(x):
    """Format floats with 6 significant digits; otherwise return the value unchanged."""
    if isinstance(x, float):
        return format(x, ".6g")
    return x

# List of annotation row identifiers to exclude from averaging.
annotation_keys = ['EWEIGHT', 'NAME', 'GWEIGHT']

# Define the root directory containing the subfolders with .pcl files.
root_dir = "/home/logs/jtorresb/yeastformer/yeast/yeast_data/data_inspection/dual_channel_extracted"

# List all subfolders in the root directory.
subfolders = [sub for sub in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, sub))]

# Iterate over each subfolder with a progress bar.
for subfolder in tqdm(subfolders, desc="Processing subfolders"):
    subfolder_path = os.path.join(root_dir, subfolder)
    # List all files ending with .pcl in the current subfolder.
    pcl_files = [file_name for file_name in os.listdir(subfolder_path) if file_name.endswith(".pcl")]
    
    for file_name in pcl_files:
        file_path = os.path.join(subfolder_path, file_name)
        try:
            # Read the file into a DataFrame (assuming tab-delimited and the first column is the index)
            df = pd.read_csv(file_path, sep="\t", index_col=0)

            # Separate annotation rows from the data rows.
            annotations = df.loc[df.index.isin(annotation_keys)]
            data = df.loc[~df.index.isin(annotation_keys)]

            # Build a dictionary grouping columns by their "base" name
            groups = {}
            for col in data.columns:
                base = extract_base(col)
                groups.setdefault(base, []).append(col)

            # Identify if any averaging is needed
            needs_averaging = any(len(cols) > 1 for cols in groups.values())

            if not needs_averaging:
                # Skip processing if no replicates are present
                continue

            # Create a new DataFrame for the averaged expression data.
            new_data = pd.DataFrame(index=data.index)
            for base, cols in groups.items():
                if len(cols) > 1:
                    new_data[base] = data[cols].mean(axis=1)
                else:
                    new_data[base] = data[cols[0]]

            # Combine the annotations with the averaged data.
            new_df = pd.concat([annotations, new_data])

            # Format all float values to 6 significant digits.
            new_df = new_df.applymap(format_float)

            # Overwrite the original file with the new DataFrame.
            new_df.to_csv(file_path, sep="\t")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("Processing complete.")

Processing subfolders: 100%|██████████| 309/309 [00:21<00:00, 14.34it/s]

Processing complete.
